In [2]:
import pandas as pd #para la lectura de datos
import redis

# Conexión a Redis local
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
#r = redis.Redis(host='157.92.26.78', port=6379, decode_responses=True)
# Decodifica automáticamente las respuestas de Redis para que no devuelva siempre bytes.

# Probar la conexión
r.ping()

True

### Modelo 1: Estado del turno

In [3]:
# Cargamos los datos
df_turnos = pd.read_csv("turno",header=None)

In [56]:
for _, row in df_turnos.iterrows(): # por cada fila del df de turnos
    r.set(                          # creamos una clave valor (turno,estado)
        f"estado_turno:{row[0]}:estado",
        row[5]
    )

In [57]:
# Ahora supongamos que una persona quiere saber si un turno no ha sido cancelado:
id_turno = 4
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))


El estado del turno 4 es: realizado


In [58]:
# Una persona cancelo un turno:
r.set(f"estado_turno:{id_turno}:estado",'cancelado')
print('El estado del turno', id_turno, 'es:', r.get(f"estado_turno:{4}:estado"))

El estado del turno 4 es: cancelado


### Modelo 2: Perfil de un paciente.

In [4]:
# Cargamos los datos
df_paciente = pd.read_csv("paciente")
#estos datos fueron tomados de una consulta en la que se joineo pacientes, persona y obra social.

In [106]:
for _, row in df_paciente.iterrows(): # por cada fila del df de pacientes
    r.hset(                           # armo un hash con el cuil como clave y me guardo todos los datos.
        f"paciente:{row['cuil']}",
        mapping={
            "nombre": row['nombre'],
            "apellido": row['apellido'],
            "genero": row['genero'],
            "grupo_sanguineo": row['grupo_sanguineo'],
            "fecha_nacimiento": row['fecha_nacimiento'],
            "telefono": row['telefono'],
            "mail": row['mail'],
            "nombre_obra": row['nombre_obra']
        }
    )

In [107]:
# supongamos que quiero saber el grupo sanguineo del paciente con el siguiente cuil:
cuil = 1234567893
print("El grupo sanguineo del paciente con cuil", cuil, "es", r.hget(f'paciente:{cuil}',"grupo_sanguineo"))

El grupo sanguineo del paciente con cuil 1234567893 es A+


In [108]:
# si quiero todos los datos del mismo paciente:
print(r.hgetall(f'paciente:{cuil}'))

{'nombre': 'nombre4', 'apellido': 'apellido4', 'genero': 'hombre', 'grupo_sanguineo': 'A+', 'fecha_nacimiento': '1999-11-10', 'telefono': '1123451237', 'mail': 'mail4@mail.cm', 'nombre_obra': 'galeno'}


In [109]:
# el paciente cambio su mail
print("El mail del paciente con cuil", cuil, "antes de actualizarlo es", r.hget(f'paciente:{cuil}',"mail"))
r.hset(f"paciente:{cuil}",mapping = {"mail": "nuevo@mail.com"})
print("El mail del paciente con cuil", cuil, "después de actualizarlo es", r.hget(f'paciente:{cuil}',"mail"))


El mail del paciente con cuil 1234567893 antes de actualizarlo es mail4@mail.cm
El mail del paciente con cuil 1234567893 después de actualizarlo es nuevo@mail.com


### Cola: Lista de pacientes de un consultorio.

In [ ]:
# Cargamos los datos
df_consultorio3 = pd.read_csv("datos_consultorio")
# estos datos fueron tomados de una consulta en la que se joineo turno con paciente y persona.
# con condiciones: numero_consultorio = 3, fecha_turno='2026-11-12', estado = 'programado'.
# nos quedamos con nombres y apellidos (ordenados por hora ascendente).
print(df_consultorio3.columns) 
# al comienzo del dia creamos un df por cada consultorio

Index(['id_turno', 'nombre', 'apellido'], dtype='object')


In [97]:
for _, row in df_consultorio3.iterrows():
    r.rpush('consultorio3',row["nombre"]+ ' ' +row["apellido"])

In [98]:
# Con r.llen vemos cuantos elementos hay en la lista
print("Cantidad de turnos pendientes:", r.llen("consultorio3"))
# Con r.lrange vemos todos los elementos de la lista
print("Pacientes pendientes:",r.lrange("consultorio3", 0, -1))

Cantidad de turnos pendientes: 2
Pacientes pendientes: ['nombre4 apellido4', 'nombre2 apellido2']


In [99]:
print("Pacientes pendientes antes de atenderlo:",r.lrange("consultorio3", 0, -1))
# Supongamos que el médico va a atender un paciente:
print(f"El siguiente paciente es {r.lpop("consultorio3")}." )
# Verificamos que el paciente haya sido eliminado de la lista:
print("Pacientes pendientes después de antenderlo:",r.lrange("consultorio3", 0, -1))

Pacientes pendientes antes de atenderlo: ['nombre4 apellido4', 'nombre2 apellido2']
El siguiente paciente es nombre4 apellido4.
Pacientes pendientes después de antenderlo: ['nombre2 apellido2']


Luego al principio del dia se volveria a agregar los pacientes para cada consultorio de la misma forma 

### Modelo 3, TTL 1: Sesiones de usuario

In [113]:
#supongamos que aproximadamente 1/4 de los pacientes accede a la web al mismo tiempo
muestra_sesiones = df_paciente.sample(frac=1, random_state=42) #tomamos aleatoriamente 25% de los pacientes


In [114]:
id_sesion = '123ABC' #esto esta para que la clave de la sesion sea más parecida a los códigos que suelen verse en las sesiones.
for i, row in muestra_sesiones.iterrows():
    r.hset(                           # armo un hash con el id de la sesion como clave y me guardo todos los datos.
        f"sesion:{id_sesion+str(i)}",
        mapping={
            "cuil" : row['cuil'],
            "nombre": row['nombre'],
            "apellido": row['apellido'],
            "genero": row['genero'],
            "grupo_sanguineo": row['grupo_sanguineo'],
            "fecha_nacimiento": row['fecha_nacimiento'],
            "telefono": row['telefono'],
            "mail": row['mail'],
            "nombre_obra": row['nombre_obra']
        }
    )
    r.expire(f"sesion:{id_sesion+str(i)}",1800) # 30 minutos

In [115]:
# Supongamos que queremos saber cuanto tiempo le queda a la sesion de un usuario:
id_sesion = '123ABC1'
r.ttl(f"sesion:{id_sesion}") # nos da cuanto tiempo le queda a la sesión


1793

In [157]:
# definimos una función que dado un id de sesion nos printee el tiempo restante o su inexistencia
def estado(nombre,id):
    if r.ttl(f"{nombre}:{id}") == -2:
        print(f"El token {id} no existe o ya expiró.")
    elif r.ttl(f"{nombre}:{id}") == -1:
        print(f"La token {id} es persistente.")
    else:
        print("El token", id,"sigue vigente por",r.ttl(f"{nombre}:{id}"),"segundos.")


In [ ]:
estado("sesion",id_sesion)

El token 123ABC1 del usuario 1234567891 sigue vigente por 1782 segundos.


In [ ]:
# Supongamos que un usuario no se mantuvo inactivo por lo que corresponde extender la sesion:
print("Antes de la actualización:")
estado("sesion",id_sesion)
r.expire(f"sesion:{id_sesion}",1800)
print("Después de la actualización:")
estado("sesion",id_sesion)


Antes de la actualización:
El token 123ABC1 del usuario 1234567891 sigue vigente por 1776 segundos.
Después de la actualización:
El token 123ABC1 del usuario 1234567891 sigue vigente por 1800 segundos.


### TTL 2: Sesión en la computadora

In [122]:
# de cada sesion nos guardamos el cuil del usuario, la hora de login y la hora de la última actividad
from datetime import datetime

id_sesion = '123ABC1'
r.hset(f'sesion_computadora:{id_sesion}',mapping={
            "cuil" : 12345678911,
            "datetime_login": datetime.now().isoformat(),
            "datetime_last_act": datetime.now().isoformat()
        })
r.expire(f"sesion_computadora:{id_sesion}",900) # 15 minutos

True

In [ ]:
# Cuando el médico realiza alguna acción:
print("Antes de la actualización:")
estado("sesion_computadora",id_sesion) # obtenemos el tiempo restante
print("El datetime es:", r.hget(f'sesion_computadora:{id_sesion}',"datetime_last_act")) # obtenemos el valor

r.hset(f'sesion_computadora:{id_sesion}',mapping={"datetime_last_act":datetime.now().isoformat()})
r.expire(f"sesion_computadora:{id_sesion}",900)

print("\nDespués de la actualización:")
estado("sesion_computadora",id_sesion) # obtenemos el nuevo tiempo restante
print("El datetime es:", r.hget(f'sesion_computadora:{id_sesion}',"datetime_last_act")) # obtenemos el nuevo valor

Antes de la actualización:
El token 123ABC1 del usuario 12345678911 sigue vigente por 898 segundos.
El datetime es: 2026-06-23T14:38:56.494114

Después de la actualización:
El token 123ABC1 del usuario 12345678911 sigue vigente por 900 segundos.
El datetime es: 2026-06-23T14:38:58.139207


### TTL 3: Reserva temporal de un turno

In [165]:
def agregar_reserva(datetime_turno,medico,id_paciente):
    clave = f"turno_reservado:medico:{medico}:{datetime_turno}"
    agregado = r.set(clave,id_paciente, nx=True, ex=600) #nx True hace que solo si guarde la clave si no existe,
    # La operación es atómica, por lo que si dos personas intentan reservar exactamente el mismo turno al mismo tiempo, sólo una lo conseguirá.
    # ex=600 es que tiene 10 minutos antes de expirar
    if not agregado: #si no se agrego
        print("El turno no se pudo reservar porque ya estaba reservado")

In [167]:
# Un paciente quiere un turno el 2026-25-12 a las 23:30:00
datetime_turno = datetime(2026, 12, 25, 11, 30)
medico = 1234567893
id_paciente = "1234567894"
agregar_reserva(datetime_turno,medico,id_paciente) # agregamos el turno por primera vez
estado("turno_reservado:medico",f"{medico}:{datetime_turno}")

agregar_reserva(datetime_turno,medico,id_paciente) # intento agregar el mismo turno otra vez



El token 1234567893:2026-12-25 11:30:00 sigue vigente por 600 segundos.
El turno no se pudo reservar porque ya estaba reservado
